# Phase 3 — Model Comparison: Exploration Notebook

This notebook explores model selection tools:
- AIC, BIC, and Akaike weights
- Likelihood ratio test for nested models
- Goodness-of-fit tests (KS, Anderson-Darling)
- Full comparison pipeline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats

from src.data_gen import generate_salary_data, generate_severance_data
from src.fitting import fit_all, fit_gamma, fit_lognormal, fit_normal
from src.model_selection import (
    compare_models, aic_weights, likelihood_ratio_test, ks_test, ad_test
)

sns.set_theme(style="whitegrid", font_scale=1.1)
SEED = 42

## 1. Full Model Comparison on Salary Data

In [ ]:
salary = generate_salary_data(500, seed=SEED)
results = fit_all(salary)
df = compare_models(results)
df

## 2. Akaike Weights Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(df["Distribution"], df["AIC_weight"], color="#2A9D8F")
ax.set_ylabel("Akaike Weight")
ax.set_title("Probability of Being the Best Model")
ax.set_ylim(0, 1)
for i, w in enumerate(df["AIC_weight"]):
    ax.text(i, w + 0.02, f"{w:.3f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 3. Likelihood Ratio Test: Exponential vs Gamma

In [ ]:
# Generate data from Gamma (alpha != 1, so Exponential is wrong)
rng = np.random.default_rng(SEED)
overtime = rng.gamma(3.0, 20.0, size=200)

# Fit both
from src.fitting import fit_gamma
gamma_result = fit_gamma(overtime)

# Exponential is Gamma with alpha=1
# LogLik of Exponential: fit with alpha fixed at 1
beta_exp = 1.0 / overtime.mean()
loglik_exp = float(np.sum(stats.expon.logpdf(overtime, scale=1.0/beta_exp)))

lrt = likelihood_ratio_test(
    loglik_restricted=loglik_exp,
    loglik_full=gamma_result.loglik,
    df=1,
)

print("Likelihood Ratio Test: Exponential vs Gamma")
print("=" * 50)
print(f"LogLik (Exponential): {loglik_exp:.2f}")
print(f"LogLik (Gamma):       {gamma_result.loglik:.2f}")
print(f"Test statistic:       {lrt.statistic:.4f}")
print(f"df:                   {lrt.df}")
print(f"p-value:              {lrt.p_value:.6f}")
print(f"Reject H0 (alpha=0.05): {lrt.reject}")
print(f"\nConclusion: Gamma fits {'significantly' if lrt.reject else 'NOT significantly'} better than Exponential")

## 4. KS and AD Tests

In [ ]:
salary = generate_salary_data(500, seed=SEED)

# Fit LogNormal (should pass) and Normal (should fail)
ln_fit = fit_lognormal(salary)
n_fit = fit_normal(salary)

# KS test for LogNormal
from scipy.stats import lognorm, norm
ks_ln = ks_test(salary, lambda x: lognorm.cdf(x, s=ln_fit.params["sigma"], scale=np.exp(ln_fit.params["mu"])))
ks_n = ks_test(salary, lambda x: norm.cdf(x, loc=n_fit.params["mu"], scale=n_fit.params["sigma"]))

# AD test
ad_ln = ad_test(salary, lambda x: lognorm.cdf(x, s=ln_fit.params["sigma"], scale=np.exp(ln_fit.params["mu"])))
ad_n = ad_test(salary, lambda x: norm.cdf(x, loc=n_fit.params["mu"], scale=n_fit.params["sigma"]))

print("Goodness-of-Fit Tests on Salary Data")
print("=" * 50)
print(f"{'Test':<8} {'Model':<12} {'Statistic':>10} {'p-value':>10} {'Decision':>10}")
print("-" * 50)
print(f"{'KS':<8} {'LogNormal':<12} {ks_ln.statistic:>10.4f} {ks_ln.p_value:>10.4f} {'reject' if ks_ln.reject else 'accept':>10}")
print(f"{'KS':<8} {'Normal':<12} {ks_n.statistic:>10.4f} {ks_n.p_value:>10.4f} {'reject' if ks_n.reject else 'accept':>10}")
print(f"{'AD':<8} {'LogNormal':<12} {ad_ln.statistic:>10.4f} {ad_ln.p_value:>10.4f} {'reject' if ad_ln.reject else 'accept':>10}")
print(f"{'AD':<8} {'Normal':<12} {ad_n.statistic:>10.4f} {ad_n.p_value:>10.4f} {'reject' if ad_n.reject else 'accept':>10}")

## 5. Effect of Sample Size on Model Selection

In [ ]:
sample_sizes = [30, 50, 100, 200, 500, 1000, 2000]
rng = np.random.default_rng(SEED)

correct_aic = []
correct_bic = []

for n in sample_sizes:
    aic_correct = 0
    bic_correct = 0
    for _ in range(100):
        data = rng.lognormal(9.1, 0.4, size=n)
        results = fit_all(data, candidates=["normal", "lognormal", "gamma"])
        df = compare_models(results)
        if df.iloc[0]["Distribution"] == "LogNormal":
            aic_correct += 1
        # BIC winner
        bic_winner = df.sort_values("BIC").iloc[0]["Distribution"]
        if bic_winner == "LogNormal":
            bic_correct += 1
    correct_aic.append(aic_correct / 100)
    correct_bic.append(bic_correct / 100)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sample_sizes, correct_aic, "o-", label="AIC selects LogNormal", linewidth=2)
ax.plot(sample_sizes, correct_bic, "s--", label="BIC selects LogNormal", linewidth=2)
ax.axhline(1.0, color="gray", linestyle=":", alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel("Sample size (n)")
ax.set_ylabel("Probability of correct selection")
ax.set_title("Model Selection Consistency: AIC vs BIC")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()